In [2]:
import pandas as pd

features_df = pd.read_csv("data/processed/features_ltr.csv")
features_df.head()


,query_id,doc_id,tfidf,bm25,common_terms,overlap_ratio,label
0,1033703,7763416,0.170245,17.398290,2,0.50,1
1,1033703,7763417,0.243869,17.860031,2,0.50,1
2,1033703,4521109,0.201760,15.901173,3,0.75,0
3,1033703,7667700,0.000000,11.077825,2,0.50,0
4,1033703,7480492,0.000000,11.012561,2,0.50,0


In [3]:
feature_cols = ["tfidf", "bm25", "common_terms", "overlap_ratio"]

X = features_df[feature_cols]
y = features_df["label"]


In [4]:
query_ids = features_df["query_id"]


In [5]:
group_sizes = query_ids.value_counts().sort_index()
group_sizes.head()


query_id
12741    9
17110    9
21603    9
22479    9
25036    9
Name: count, dtype: int64

In [6]:
from sklearn.model_selection import train_test_split

unique_queries = query_ids.unique()

train_q, test_q = train_test_split(
    unique_queries,
    test_size=0.2,
    random_state=42
)

train_mask = query_ids.isin(train_q)
test_mask = query_ids.isin(test_q)

X_train = X[train_mask]
y_train = y[train_mask]
X_test = X[test_mask]
y_test = y[test_mask]


In [7]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


/Users/divyanshunagar/Desktop/RANKING_SYSTEM/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/divyanshunagar/Desktop/RANKING_SYSTEM/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/divyanshunagar/Desktop/RANKING_SYSTEM/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept


LogisticRegression(max_iter=1000)

In [8]:
test_scores = model.predict_proba(X_test)[:, 1]


/Users/divyanshunagar/Desktop/RANKING_SYSTEM/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/divyanshunagar/Desktop/RANKING_SYSTEM/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/divyanshunagar/Desktop/RANKING_SYSTEM/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [9]:
test_results = features_df[test_mask].copy()
test_results["score"] = test_scores


In [10]:
ranked_results = (
    test_results
    .sort_values(["query_id", "score"], ascending=[True, False])
)
ranked_results.head()


,query_id,doc_id,tfidf,bm25,common_terms,overlap_ratio,label,score
301,22479,7745789,0.122240,9.189716,1,0.25,0,0.154910
305,22479,7555802,0.103680,8.663744,1,0.25,0,0.130586
303,22479,7208493,0.064217,8.862256,1,0.25,0,0.103767
299,22479,7710581,0.067369,13.235450,2,0.50,1,0.102185
307,22479,7368724,0.046749,7.737846,1,0.25,0,0.081543


In [12]:
from ranx import Qrels, Run, evaluate

qrels_test = {}
run_dict = {}

for qid, group in ranked_results.groupby("query_id"):
    qid = str(qid)   # ensure string

    qrels_test[qid] = {}
    run_dict[qid] = {}

    for _, row in group.iterrows():
        doc_id = str(row["doc_id"])        # STRING
        label  = int(row["label"])         # INT
        score  = float(row["score"])       # FLOAT

        qrels_test[qid][doc_id] = label
        run_dict[qid][doc_id] = score


metrics_ltr = evaluate(
    Qrels(qrels_test),
    Run(run_dict),
    ["ndcg@10", "map"]
)

metrics_ltr


{'ndcg@10': np.float64(0.827977807477571),
 'map': np.float64(0.7747222222222222)}

FAILURE ANALYSIS

In [13]:
def bm25_top_docs(query, k=5):
    scores = bm25.get_scores(query.lower().split())
    ranked = sorted(zip(doc_ids, scores), key=lambda x: x[1], reverse=True)
    return ranked[:k]


In [14]:
ranked_results


,query_id,doc_id,tfidf,bm25,common_terms,overlap_ratio,label,score
301,22479,7745789,0.122240,9.189716,1,0.25,0,0.154910
305,22479,7555802,0.103680,8.663744,1,0.25,0,0.130586
303,22479,7208493,0.064217,8.862256,1,0.25,0,0.103767
299,22479,7710581,0.067369,13.235450,2,0.50,1,0.102185
307,22479,7368724,0.046749,7.737846,1,0.25,0,0.081543
...,...,...,...,...,...,...,...,...
1349,1101868,6792346,0.231943,15.304784,2,0.50,0,0.325500
1343,1101868,7556859,0.193768,17.414761,2,0.50,0,0.322441
1348,1101868,7326875,0.200629,15.925939,2,0.50,0,0.293667
1347,1101868,7556541,0.182222,16.461111,2,0.50,0,0.280060


In [16]:
from beir.datasets.data_loader import GenericDataLoader

data_path = "data/raw/msmarco"
corpus, queries, qrels = GenericDataLoader(data_path).load(split="dev")


/Users/divyanshunagar/Desktop/RANKING_SYSTEM/venv/lib/python3.9/site-packages/beir/datasets/data_loader.py:8: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
100%|█████████████████████████████| 8841823/8841823 [00:16<00:00, 543082.84it/s]


In [23]:
relevant_doc_ids = set()
for qid, docs in qrels.items():
    for d in docs:
        relevant_doc_ids.add(d)

filtered_corpus = {d: corpus[d] for d in relevant_doc_ids}


In [24]:
from rank_bm25 import BM25Okapi

doc_ids = list(filtered_corpus.keys())
docs_text = [filtered_corpus[d]["text"] for d in doc_ids]

tokenized_docs = [text.lower().split() for text in docs_text]

bm25 = BM25Okapi(tokenized_docs)


In [25]:
def bm25_top_docs(query, k=5):
    scores = bm25.get_scores(query.lower().split())
    ranked = sorted(zip(doc_ids, scores), key=lambda x: x[1], reverse=True)
    return ranked[:k]


In [28]:
sample_qid = ranked_results["query_id"].iloc[0]
query_text = queries[str(sample_qid)]

print("QUERY:", query_text)

print("\nBM25 TOP RESULTS")
for doc_id, score in bm25_top_docs(query_text, k=5):
    print("Label:", qrels.get(str(sample_qid), {}).get(doc_id, 0))
    print(filtered_corpus[doc_id]["text"][:150])
    print("-"*80)

print("\nLTR TOP RESULTS")

ltr_group = ranked_results[ranked_results["query_id"] == sample_qid].head(5)

for _, row in ltr_group.iterrows():
    doc_id = str(int(row["doc_id"]))   # ✅ FIX
    print("Label:", row["label"], "Score:", row["score"])
    print(filtered_corpus[doc_id]["text"][:150])
    print("-" * 80)


QUERY: are florida mangroves protected

BM25 TOP RESULTS
Label: 1
Thomson Reuters Handout of red and black mangrove in Miami. By Zachary Fagenson. MIAMI, Fla. (Reuters) - New revelations that a long strip of protecte
--------------------------------------------------------------------------------
Label: 0
Florida Marketing Organization has a dedicated focus in Florida to make sure these offerings are suitable for our clients needs. FMO teams up with the
--------------------------------------------------------------------------------
Label: 0
Florida Gas Tax. The Florida excise tax on gasoline is 4.00Â¢ per gallon, one of the highest gas taxes in the country. Florida's excise tax on gasolin
--------------------------------------------------------------------------------
Label: 0
HR Guide to Employment Law: A practical compliance reference manual covering 14 topics, including overtime. Discrimination and harassment. Employees, 
------------------------------------------------------------

“In some factoid queries with strong lexical signals, BM25 outperformed the LTR model.
The LTR over-penalized exact matches due to feature interactions, pushing the relevant document lower.
This highlights the importance of hybrid ranking strategies.”

In [37]:
import os
os.makedirs("data/processed", exist_ok=True)

ranked_results.to_csv(
    "data/processed/ltr_ranked_results.csv",
    index=False
)

print("Saved ltr_ranked_results.csv")


Saved ltr_ranked_results.csv
